# JSON - JavaScript

All 9 JavaScript examples from [docs/json.md](https://platob.github.io/yggdryl/json/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const value = { symbol: 'AAPL', quantity: 100 }

const encoded = json.dumps(value)
assert.equal(encoded.toString(), '{"symbol":"AAPL","quantity":100}')
assert.deepEqual(json.loads(encoded), value)
assert.deepEqual(json.loads('{"symbol":"AAPL","quantity":100}'), value)
assert.equal(json.loads(encoded).symbol, 'AAPL')

## Values JSON has no syntax for

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const value = { payload: Buffer.from([0, 1, 255]), big: 2n ** 127n, ratio: NaN }

const encoded = json.dumps(value)
assert.ok(encoded.includes('"$yggdryl"'))

const decoded = json.loads(encoded)
assert.deepEqual(decoded.payload, Buffer.from([0, 1, 255]))
assert.equal(decoded.big, 2n ** 127n)
assert.ok(Number.isNaN(decoded.ratio))

## Readers and writers

In [ ]:
const assert = require('node:assert/strict')
const { Readable, Writable } = require('node:stream')
const { json } = require('yggdryl')

const value = { symbol: 'AAPL' }

async function main() {
  const chunks = []
  const target = new Writable({
    write(chunk, _encoding, done) {
      chunks.push(Buffer.from(chunk))
      done()
    },
  })

  await json.dumpStream(value, target)
  assert.deepEqual(await json.loadStream(Readable.from(chunks)), value)

  assert.throws(
    () => json.loads('{"symbol":"AAPL"} 42'),
    /invalid json data at byte 18: trailing characters after JSON value/,
  )
}

main()

## Newline-delimited JSON

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const rows = [{ id: 1 }, { id: 2 }]

const encoded = json.dumpAll(rows)
assert.equal(encoded.toString(), '{"id":1}\n{"id":2}\n')
assert.deepEqual(json.loadsAll(encoded), rows)

// Blank and CRLF-terminated lines are skipped; two values on one are not.
assert.deepEqual(json.loadsAll('{"id":1}\r\n\n{"id":2}\n'), rows)
assert.throws(
  () => json.loadsAll('{"id":1} {"id":2}\n'),
  /invalid json data at byte 9: trailing characters after JSON value/,
)

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

let pulls = 0
async function* chunks() {
  pulls += 1
  yield '{"id":1}\n{"i'
  pulls += 1
  yield 'd":2}\n'
}

async function main() {
  const rows = json.loadAllStream(chunks())[Symbol.asyncIterator]()

  assert.deepEqual((await rows.next()).value, { id: 1 })
  assert.equal(pulls, 1)
  assert.deepEqual((await rows.next()).value, { id: 2 })
  assert.equal(pulls, 2)
  assert.equal((await rows.next()).done, true)
}

main()

## Limits

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

let deep = 0
for (let index = 0; index < 60; index += 1) deep = [deep]
assert.throws(() => json.dumps(deep), /exceeds maxDepth 48/)
assert.throws(() => json.dumps(deep, { maxDepth: 100 }), /between 1 and 48/)

let shallow = 0
for (let index = 0; index < 20; index += 1) shallow = [shallow]
assert.throws(
  () => json.loads(json.dumps(shallow), { maxDepth: 16 }),
  /invalid json data at byte 16: nesting depth limit exceeded/,
)

function* many() {
  for (let index = 0; index < 1025; index += 1) yield { id: index }
}
assert.throws(() => json.dumpAll(many()), /codec collection exceeds the 1024-document limit/)

## Failures carry a byte offset

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

assert.throws(
  () => json.loads('{"symbol":"AAPL","symbol":"MSFT"}'),
  /invalid json data at byte 17: JSON object contains a duplicate key/,
)
assert.throws(
  () => json.loadsAll('{"id":1}\n{bad}\n'),
  /invalid json data at byte 10: JSON object key must be a string/,
)

## A compound filename carries the coding

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { pathToFileURL } = require('node:url')
const { json } = require('yggdryl')

const directory = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-json-'))
try {
  const file = path.join(directory, 'trades.json')

  json.dump({ symbol: 'AAPL' }, file)
  assert.deepEqual(json.load(file), { symbol: 'AAPL' })
  assert.deepEqual(json.load(pathToFileURL(file)), { symbol: 'AAPL' })

  // A string that is not an existing file is content, not a location.
  assert.deepEqual(json.load('{"symbol":"AAPL"}'), { symbol: 'AAPL' })
} finally {
  fs.rmSync(directory, { force: true, recursive: true })
}

## Placeholders

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const value = json.loads('{"port": "{{ PORT }}"}', { placeholders: { PORT: 8080 } })
assert.equal(value.port, 8080)